# Stage 00b2 — Descriptions CSV + Figure Cropping & Label Matching

**What this notebook does:**

1. Exports all *Brief Description of the Drawings* texts from the PatSeer Excel
   into a single `data/descriptions.csv`.

2. Detects and crops every sub-figure on each drawing sheet using
   **DocLayout-YOLO** (document layout detector) + **EasyOCR** (label reader) —
   free, local, GPU-accelerated.

The matching logic:
1. DocLayout-YOLO detects `figure` and `figure_caption` regions on each sheet
2. For each figure box, EasyOCR tries to read the label via a cascade of passes:
   top-left corner → top-right corner → below strip (350 px) → side margins
3. Each crop is saved as `_F<label>.png` (matched) or `_Fu.png` (needs review)
4. FAT files are always copied whole as `_Fu`

**Output naming:**

| Label source | Output filename | Meaning |
|---|---|---|
| OCR label match | `US…_F1A.png` | matched to FIG. 1A |
| Unmatched / FAT / error | `US…_Fu.png` | needs review |

Output is written to `matched/<patent_id>/` — raw files are never modified.

**Where it fits in the pipeline:**
```
00a  (PatSeer download)
 ↓
00b1 (grouping + batch assignment → batches.xlsx)
 ↓
00b2 ←  YOU ARE HERE  (run once per batch: descriptions CSV + DocLayout-YOLO → _F / _Fu labels)
 ↓
01   (human review for _Fu files)
 ↓
02   (pad + resize to 518×518)
```

---

| Cell | What it does |
|------|------|
| 1 | CUDA check |
| 2 | Load config + Excel, save descriptions.csv |
| 2b | Batch selector — set BATCH_INDEX and run |
| 3 | Build DocLayout-YOLO + EasyOCR engine (once per session) |
| 4 | Run pipeline over selected batch |
| 5 | Save review CSV + show flagged crops |

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print("Is CUDA active inside Jupyter?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Connected GPU Model:", torch.cuda.get_device_name(0))

In [ ]:
import sys
import os
import pandas as pd
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent
src_dir = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config_loader import load_config
from extractor import load_patseer_excel, save_descriptions_csv
import doclayout_matcher as dm

cfg = load_config()

print("Config loaded. Key paths:")
print("  raw_images :", cfg["paths"]["raw_images"])
print("  matched    :", cfg["paths"]["matched"])
print("  data       :", cfg["paths"]["data"])

In [ ]:
excel_path = Path(cfg["paths"]["patseer_excel"])
print(f"Reading PatSeer data from: {excel_path.name}...")
patseer_index = load_patseer_excel(excel_path)

desc_csv_path = Path(cfg["paths"]["data"]) / "descriptions.csv"
save_descriptions_csv(patseer_index, desc_csv_path)

df = pd.read_csv(desc_csv_path)
print(f"descriptions.csv saved — {len(df)} records.")

In [ ]:
# ── Cell 2b — Batch selector ──────────────────────────────────────────────────
# Loads batch assignments from data/batches.xlsx (produced by 00b1_grouping).
# Set BATCH_INDEX to run one batch at a time (0-based: 0, 1, 2, ...).
# Set BATCH_INDEX = None to run all patents.
#
# Run 00b1_grouping first — it creates batches.xlsx.

import pandas as pd
from pathlib import Path

BATCH_INDEX = 1      # ← change this to run different batches

batches_xlsx = Path(cfg["paths"]["data"]) / "batches.xlsx"
if not batches_xlsx.exists():
    raise FileNotFoundError(
        f"batches.xlsx not found at {batches_xlsx}\n"
        "Run 00b1_grouping.ipynb first to generate batch assignments."
    )

xl = pd.ExcelFile(batches_xlsx)
batch_sheet_names = [s for s in xl.sheet_names if s.startswith("Batch_")]
all_batches = [xl.parse(s, dtype=str) for s in batch_sheet_names]

print(f"Total patents   : {len(df)}")
print(f"Batches found   : {len(all_batches)}  (from {batches_xlsx.name})")
for i, b in enumerate(all_batches):
    print(f"  Batch {i}: {len(b)} patents")

if BATCH_INDEX is None:
    run_df = df
    print(f"\nRunning ALL {len(run_df)} patents.")
else:
    if BATCH_INDEX >= len(all_batches):
        raise ValueError(f"BATCH_INDEX {BATCH_INDEX} out of range — only {len(all_batches)} batches exist.")
    batch_df = all_batches[BATCH_INDEX]
    # batches.xlsx uses 'patent_id' column; filter df to only those patent_ids
    batch_ids = set(batch_df["patent_id"].dropna().str.strip())
    run_df = df[df["patent_id"].isin(batch_ids)].reset_index(drop=True)
    print(f"\nRunning Batch {BATCH_INDEX}: {len(run_df)} patents.")

In [ ]:
# Engines are now built inside Cell 4 (dual-GPU). This cell is kept as a
# quick CUDA sanity check — run it to confirm both GPUs are visible.
import torch
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i}  {torch.cuda.get_device_name(i)}  "
          f"{torch.cuda.get_device_properties(i).total_memory // 1024**2} MB")


In [ ]:
import re, time, shutil, json

raw_dir     = Path(cfg["paths"]["raw_images"])
matched_dir = Path(cfg["paths"]["matched"])
triage_dir  = Path(cfg["paths"]["triage"])
matched_dir.mkdir(parents=True, exist_ok=True)

scan_limit = cfg["extractor"].get("scan_limit", None)
if scan_limit:
    run_df = run_df.head(scan_limit)
print(f"Running on {len(run_df)} patents (scan_limit={scan_limit})...")

# ── Sheet/triage helpers (unchanged) ─────────────────────────────────────────
_CLEAN_RE   = re.compile(r"[^A-Za-z0-9]")
_NUM_SUFFIX = re.compile(r"_\d+$")
_DL_SUFFIX  = re.compile(r"PAFP$|PAF$", re.IGNORECASE)
_KIND_CODES = ["A1","A2","A3","B1","B2","C1","U1"]

def _core(pid):
    p = _NUM_SUFFIX.sub("", pid)
    c = _CLEAN_RE.sub("", p).upper()
    c = _DL_SUFFIX.sub("", c)
    for sfx in _KIND_CODES:
        if c.endswith(sfx):
            return c[:-len(sfx)]
    return c

folder_map = {_core(p.name): p for p in raw_dir.iterdir() if p.is_dir()}

_triage_excluded_cache: dict = {}
def _triage_excluded_files(patent_id):
    if patent_id in _triage_excluded_cache:
        return _triage_excluded_cache[patent_id]
    json_path = triage_dir / f"{patent_id}.json"
    excluded = set()
    if json_path.exists():
        try:
            data = json.loads(json_path.read_text())
            excluded = {fig["file"] for fig in data.get("figures", [])
                        if fig.get("keep") is False and fig.get("locked") is True}
        except Exception as e:
            print(f"  ⚠  Could not read triage JSON for {patent_id}: {e}")
    _triage_excluded_cache[patent_id] = excluded
    return excluded

_NON_SHEET = re.compile(r"manifest|thumbnail|cover|abstract|front.?page", re.IGNORECASE)
_SHEET_RE  = re.compile(r"""
    (?:
        _[Dd]\d{3,}          |   # _D00001
        PAFP_img\d           |   # US...PAFP_img1
        PAF_img\d            |   # WO...PAF_img1
        _img[af]?\d          |   # _img1 _imgf1 _imgaf1
        fig_\d               |   # fig_01
        record__fig_\d       |   # record__fig_01
        ^img[af]?\d          |   # imgf0001 imgaf001
        ^pat\d               |   # pat1.png
        ^FT_\d               |   # FT_1.png
        ^HDA\d               |   # HDA1.png
        ^\d+\.               |   # 1.png 2.png
        ^srep\d              |   # srep1.png
        sN_img\d                 # SDN...SN_img1
    )
""", re.VERBOSE | re.IGNORECASE)

def is_sheet(f):
    if f.suffix.lower() != ".png": return False
    if _NON_SHEET.search(f.name): return False
    return bool(_SHEET_RE.search(f.name))

# ── Run parallel processing (two subprocesses, one per GPU) ──────────────────
import torch
print(f"GPUs available: {torch.cuda.device_count()} — each worker builds its own engine")
t0 = time.time()

rows, triage_skipped_total = dm.process_patents_parallel(
    patent_rows            = run_df["patent_id"],
    folder_map             = folder_map,
    matched_dir            = matched_dir,
    triage_dir             = triage_dir,
    engines                = None,
    is_sheet_fn            = None,
    triage_excluded_fn     = None,
    cfg                    = cfg,
)

elapsed = time.time() - t0
results_df = pd.DataFrame(rows)

crops_csv = Path(cfg["paths"]["data"]) / "crops_mapping.csv"
results_df.to_csv(crops_csv, index=False)

print(f"\nDone in {elapsed:.0f}s  ({elapsed/max(1,len(run_df)):.1f}s/patent)")
print(f"Total crops : {len(results_df)}")
if triage_skipped_total:
    print(f"Skipped (triage-rejected) sheets: {triage_skipped_total}")
if not results_df.empty:
    labelled_n = (results_df["needs_review"] == False).sum()
    print(f"Auto-labelled: {labelled_n}/{len(results_df)} = {100*labelled_n/len(results_df):.0f}%")
    print(results_df.groupby("method")["output"].count())
    if "review_hint" in results_df.columns:
        hint_n = (results_df["review_hint"] == "possible_multi_fig").sum()
        if hint_n:
            print(f"\nFlagged as possible multi-figure merges: {hint_n}")

In [ ]:
if results_df.empty or "needs_review" not in results_df.columns:
    print("No crops produced — nothing to review.")
else:
    flagged_df = results_df[results_df["needs_review"] == True]
    review_csv = Path(cfg["paths"]["data"]) / "needs_human_review.csv"
    flagged_df.to_csv(review_csv, index=False)
    print(f"Saved {len(flagged_df)} flagged crops → {review_csv.name}")
    print(f"(Pass these to Step 01 — human review notebook)")
    if not flagged_df.empty:
        display(flagged_df[["patent_id","original","output","label","method"]].head(20))

In [ ]:
# ── Cell 5b — Compute crop_quality flags from already-saved crops ───────────
# Pure post-processing: inspects PNGs already written to matched/, never
# touches raw/, never re-runs YOLO/EasyOCR. Appends crop_quality to
# crops_mapping.csv and overwrites it in place.
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import sys
from pathlib import Path as _Path
_notebook_dir  = _Path(__import__("os").getcwd())
_project_root  = _notebook_dir.parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
from reviewer import resolve_patent_image_dir

crops_csv   = Path(cfg["paths"]["data"]) / "crops_mapping.csv"
matched_dir = Path(cfg["paths"]["matched"])
quality_df  = pd.read_csv(crops_csv)

BLANK_BRIGHTNESS_THRESH = 248
MIN_CROP_DIM_PX         = 80
LOW_CONF_THRESH         = 0.40

has_conf = "conf" in quality_df.columns
has_hint = "review_hint" in quality_df.columns

# matched/ folders are named "{patent_id}_{record_number}", not the bare
# patent_id — resolve via the same helper reviewer.py/run_stage01() use, and
# cache per patent_id since this is called once per row.
_patent_dir_cache: dict = {}

def _patent_dir(patent_id: str) -> Path:
    if patent_id not in _patent_dir_cache:
        _patent_dir_cache[patent_id] = resolve_patent_image_dir(matched_dir, str(patent_id))
    return _patent_dir_cache[patent_id]


def _crop_quality(row) -> str:
    img_path = _patent_dir(row["patent_id"]) / str(row["output"])
    if not img_path.exists():
        return "missing"
    try:
        with Image.open(img_path) as im:
            gray = np.asarray(im.convert("L"))
    except Exception:
        return "missing"

    h, w = gray.shape[:2]
    if float(gray.mean()) > BLANK_BRIGHTNESS_THRESH:
        return "blank"
    if min(h, w) < MIN_CROP_DIM_PX:
        return "too_small"
    if has_conf and pd.notna(row.get("conf")) and float(row["conf"]) < LOW_CONF_THRESH:
        return "low_conf"
    if has_hint and row.get("review_hint") == "possible_multi_fig":
        return "merged"
    return ""


quality_df["crop_quality"] = quality_df.apply(_crop_quality, axis=1)
quality_df.to_csv(crops_csv, index=False)

summary = quality_df["crop_quality"].fillna("").value_counts()
print(f"crop_quality summary ({len(quality_df)} crops):\n")
print(f"  {'crop_quality':<14}| count")
print(f"  {'-'*14}|------")
for label in ["", "blank", "too_small", "low_conf", "merged", "missing"]:
    display_label = "(empty)" if label == "" else label
    print(f"  {display_label:<14}| {int(summary.get(label, 0))}")

In [ ]:
# ── Cell 5c — Interactive crop reviewer (Keep / Relabel / Reject) ───────────
# Shows only crops that need human attention (needs_review == True OR
# crop_quality != ""). Every action is written to crops_mapping.csv
# immediately, so progress survives kernel restarts. Files are renamed in
# matched/ only on Relabel — raw/ is never touched, YOLO/EasyOCR never re-run.
import re
import base64
import pandas as pd
from pathlib import Path
from functools import partial
import ipywidgets as widgets
from IPython.display import display, clear_output
import sys
from pathlib import Path as _Path
_notebook_dir  = _Path(__import__("os").getcwd())
_project_root  = _notebook_dir.parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
from reviewer import resolve_patent_image_dir

crops_csv   = Path(cfg["paths"]["data"]) / "crops_mapping.csv"
matched_dir = Path(cfg["paths"]["matched"])

results_df = pd.read_csv(crops_csv)
if "crop_quality" not in results_df.columns:
    results_df["crop_quality"] = ""
results_df["crop_quality"] = results_df["crop_quality"].fillna("")

# matched/ folders are named "{patent_id}_{record_number}" — resolve via the
# same helper reviewer.py/run_stage01() use, cached per patent_id.
_patent_dir_cache: dict = {}

def _patent_dir(patent_id: str) -> Path:
    if patent_id not in _patent_dir_cache:
        _patent_dir_cache[patent_id] = resolve_patent_image_dir(matched_dir, str(patent_id))
    return _patent_dir_cache[patent_id]


_BADGE_COLOR = {
    "blank": "#cf222e", "too_small": "#cf222e", "missing": "#cf222e",
    "low_conf": "#9a6700", "merged": "#9a6700",
}

flagged_mask  = (results_df["needs_review"] == True) | (results_df["crop_quality"] != "")
flagged_idxs  = set(results_df.index[flagged_mask])
flagged_patents = results_df.loc[flagged_mask, "patent_id"].drop_duplicates().tolist()
resolved_idxs = set()

state  = {"patent_idx": 0}
output = widgets.Output()

prev_btn = widgets.Button(description="◄ Prev")
next_btn = widgets.Button(description="Next ►")


def _b64(path: Path) -> str:
    return base64.b64encode(path.read_bytes()).decode()


def _badge_for(row) -> tuple[str, str]:
    q = row["crop_quality"]
    if q in _BADGE_COLOR:
        return _BADGE_COLOR[q], q
    if bool(row["needs_review"]):
        return "#c9b400", "needs_review"
    return "#888", q or "ok"


def _do_action(idx, *, new_quality=None, new_output=None, new_label=None, needs_review=None):
    if new_quality is not None:
        results_df.loc[idx, "crop_quality"] = new_quality
    if new_output is not None:
        results_df.loc[idx, "output"] = new_output
    if new_label is not None:
        results_df.loc[idx, "label"] = new_label
    if needs_review is not None:
        results_df.loc[idx, "needs_review"] = needs_review
    resolved_idxs.add(idx)
    results_df.to_csv(crops_csv, index=False)
    render()


def on_keep(idx, _btn):
    _do_action(idx, new_quality="")


def on_reject(idx, _btn):
    _do_action(idx, new_quality="rejected")


def on_recrop(idx, _btn):
    _do_action(idx, new_quality="re_crop")


def on_toggle_relabel(box, _btn):
    box.layout.display = "flex" if box.layout.display == "none" else "none"


def on_apply_relabel(idx, text_input, _btn):
    new_label = text_input.value.strip()
    if not new_label:
        return
    row = results_df.loc[idx]
    old_output = str(row["output"])
    if old_output.endswith("_Fu.png"):
        new_output = old_output[: -len("_Fu.png")] + f"_F{new_label}.png"
    else:
        new_output = re.sub(r"\.png$", f"_F{new_label}.png", old_output)

    patent_dir = _patent_dir(row["patent_id"])
    old_path = patent_dir / old_output
    new_path = patent_dir / new_output
    if old_path.exists():
        old_path.rename(new_path)

    _do_action(idx, new_quality="", new_output=new_output, new_label=new_label, needs_review=False)


def build_card(idx, row) -> widgets.VBox:
    img_path = _patent_dir(row["patent_id"]) / str(row["output"])
    badge_color, badge_text = _badge_for(row)
    label_text = row["label"] if pd.notna(row["label"]) else "Fu"
    img_b64 = _b64(img_path) if img_path.exists() else ""

    card_html = widgets.HTML(
        f"<div style='border:2px solid {badge_color}; border-radius:6px; padding:6px; "
        f"text-align:center;'>"
        f"<img src='data:image/png;base64,{img_b64}' "
        f"style='max-width:220px; max-height:200px; object-fit:contain;'/><br>"
        f"<span style='background:{badge_color}; color:white; padding:1px 6px; "
        f"border-radius:4px; font-size:11px;'>{badge_text}</span><br>"
        f"<b>{label_text}</b><br>"
        f"<span style='color:grey; font-size:10px;'>{row['crop_quality'] or 'needs_review'}</span>"
        f"</div>"
    )

    keep_btn    = widgets.Button(description="✓ Keep", button_style="success", layout=widgets.Layout(width="70px"))
    relabel_btn = widgets.Button(description="✎ Relabel", layout=widgets.Layout(width="80px"))
    reject_btn  = widgets.Button(description="✗ Reject", button_style="danger", layout=widgets.Layout(width="75px"))
    recrop_btn  = widgets.Button(description="✂ Re-crop", button_style="warning", layout=widgets.Layout(width="80px"))

    text_input = widgets.Text(placeholder="new label e.g. 3B", layout=widgets.Layout(width="120px"))
    apply_btn  = widgets.Button(description="Apply", button_style="info", layout=widgets.Layout(width="60px"))
    relabel_box = widgets.HBox([text_input, apply_btn], layout=widgets.Layout(display="none"))

    keep_btn.on_click(partial(on_keep, idx))
    reject_btn.on_click(partial(on_reject, idx))
    recrop_btn.on_click(partial(on_recrop, idx))
    relabel_btn.on_click(partial(on_toggle_relabel, relabel_box))
    apply_btn.on_click(partial(on_apply_relabel, idx, text_input))

    buttons_row = widgets.HBox([keep_btn, relabel_btn, reject_btn, recrop_btn])
    return widgets.VBox([card_html, buttons_row, relabel_box])


def change_patent(delta, _btn=None):
    state["patent_idx"] = max(0, min(state["patent_idx"] + delta, len(flagged_patents) - 1))
    render()


prev_btn.on_click(partial(change_patent, -1))
next_btn.on_click(partial(change_patent, 1))


def render():
    with output:
        clear_output(wait=True)
        if not flagged_idxs:
            display(widgets.HTML("<i>No crops need review — crops_mapping.csv is clean.</i>"))
            return

        pid = flagged_patents[state["patent_idx"]]
        recrop_count = int((results_df["crop_quality"] == "re_crop").sum())
        header = widgets.HTML(
            f"<h3>{len(flagged_idxs)} crops need attention ({len(flagged_patents)} patents)</h3>"
            f"<p>Reviewed: {len(resolved_idxs)} / {len(flagged_idxs)} &middot; Re-crop flagged: {recrop_count}</p>"
        )
        nav = widgets.HBox([
            prev_btn,
            widgets.Label(f"patent {state['patent_idx'] + 1}/{len(flagged_patents)} — {pid}"),
            next_btn,
        ])

        remaining = sorted(flagged_idxs - resolved_idxs)
        patent_rows = results_df.loc[[i for i in remaining if results_df.loc[i, "patent_id"] == pid]]

        if patent_rows.empty:
            grid = widgets.HTML("<i>All flagged crops for this patent are reviewed ✓</i>")
        else:
            cards = [build_card(idx, row) for idx, row in patent_rows.iterrows()]
            grid = widgets.GridBox(
                cards,
                layout=widgets.Layout(
                    grid_template_columns="repeat(3, 1fr)",
                    grid_gap="12px",
                    max_height="600px",
                    overflow_y="auto",
                ),
            )

        display(widgets.VBox([header, nav, grid]))


render()
display(output)

---
## Evaluation — inspect results inline

**Cell 6** — Summary stats (counts, labelling rate, method breakdown per patent)  
**Cell 7** — Visual grid: all crops for every patent, annotated with label + method  
**Cell 8** — Flagged-only reviewer: show just the `_Fu` crops that need human attention

In [ ]:
# ── Cell 6 — Summary stats ────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

crops_csv = Path(cfg["paths"]["data"]) / "crops_mapping.csv"
results_df = pd.read_csv(crops_csv)

if results_df.empty:
    print("crops_mapping.csv is empty — run the pipeline first (Cell 4).")
else:
    total      = len(results_df)
    labelled_n = (results_df["needs_review"] == False).sum()
    flagged_n  = (results_df["needs_review"] == True).sum()

    print(f"{'Total crops':<25} {total}")
    print(f"{'Auto-labelled (_F)':<25} {labelled_n}  ({100*labelled_n/total:.0f}%)")
    print(f"{'Needs review (_Fu)':<25} {flagged_n}  ({100*flagged_n/total:.0f}%)")
    print()

    # Per-patent breakdown
    per_patent = (
        results_df
        .groupby("patent_id")
        .agg(
            sheets   = ("original", "nunique"),
            crops    = ("output", "count"),
            labelled = ("needs_review", lambda x: (x == False).sum()),
            flagged  = ("needs_review", lambda x: (x == True).sum()),
        )
        .assign(pct=lambda d: (d["labelled"] / d["crops"] * 100).round(0).astype(int))
        .rename(columns={"pct": "labelled_%"})
        .sort_values("labelled_%")
    )
    display(per_patent)

    print()
    print("Method breakdown:")
    display(results_df.groupby("method")["output"].count().rename("crops").to_frame())

In [ ]:
# ── Cell 7 — Crops viewer (3-column HTML grid, ← → patent navigation) ────────
import re, base64, textwrap
import pandas as pd
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

matched_dir = Path(cfg["paths"]["matched"])
crops_df    = pd.read_csv(Path(cfg["paths"]["data"]) / "crops_mapping.csv")
desc_df     = pd.read_csv(Path(cfg["paths"]["data"]) / "descriptions.csv")
desc_lookup = desc_df.set_index("patent_id")["description_of_drawings"].to_dict()

_FIG_RE = re.compile(
    r"FIG\.?\s*(\d+\s*[A-Za-z]?)\s+(?:and\s+FIG\.?\s*\d+\s*[A-Za-z]?\s+)?(?:is|are|show|illustrate|depict)\b",
    re.IGNORECASE,
)

def parse_fig_descriptions(text):
    if not isinstance(text, str): return {}
    text = re.sub(r"\s*\n\s*", " ", text)
    parts = re.split(r"(?=FIG\.?\s*\d)", text, flags=re.IGNORECASE)
    result = {}
    for part in parts:
        m = _FIG_RE.search(part)
        if m:
            key = re.sub(r"\s+", "", m.group(1)).upper()
            result.setdefault(key, re.sub(r"\s{2,}", " ", part).strip()[:300])
    return result

def _b64(path):
    return base64.b64encode(path.read_bytes()).decode()

patents = list(crops_df["patent_id"].unique())
if not patents:
    print("No crops yet — run the pipeline (Cell 4) first.")
else:
    state    = {"idx": 0}
    counter  = widgets.HTML()
    prev_btn = widgets.Button(description="◀ Prev", button_style="info",
                              layout=widgets.Layout(width="100px"))
    next_btn = widgets.Button(description="Next ▶", button_style="info",
                              layout=widgets.Layout(width="100px"))
    out      = widgets.Output()

    # Build a lookup: output filename → absolute path, searching all matched locations
    base_dir = Path(cfg["paths"]["base"])
    all_matched_roots = [matched_dir] + sorted(base_dir.glob("*/matched"))
    _crop_file_index: dict = {}
    for root in all_matched_roots:
        if not root.exists(): continue
        for patent_folder in root.iterdir():
            if not patent_folder.is_dir(): continue
            for f in patent_folder.iterdir():
                _crop_file_index[f.name] = f

    def _find_crop(filename: str):
        return _crop_file_index.get(filename)

    def render(idx):
        patent_id = patents[idx]
        grp       = crops_df[crops_df["patent_id"] == patent_id]
        fig_descs = parse_fig_descriptions(desc_lookup.get(patent_id, ""))
        out_dir   = None  # unused now — we use _find_crop() per file

        n_ok = (grp["needs_review"] == False).sum()
        n_fu = (grp["needs_review"] == True).sum()

        cards = ""
        for _, row in grp.iterrows():
            img_path = _find_crop(row["output"])
            img_tag  = ""
            if img_path and img_path.exists():
                img_tag = f'<img src="data:image/png;base64,{_b64(img_path)}" style="max-width:100%;max-height:260px;object-fit:contain;"/>'

            label_key = str(row["label"]).upper() if pd.notna(row["label"]) else None
            hint      = row.get("review_hint") or ""

            if not row["needs_review"]:
                badge_bg, badge_col = "#d4edda", "#155724"
                badge_txt = f"&#10003; MATCHED &nbsp;&nbsp; FIG.&nbsp;{row['label']}"
                desc_html = ""
                if label_key and label_key in fig_descs:
                    desc_html = f'<p style="font-size:10px;color:#155724;margin:4px 0 0 0;">{fig_descs[label_key]}</p>'
            elif hint == "possible_multi_fig":
                badge_bg, badge_col = "#fff3cd", "#856404"
                badge_txt = "&#9888; POSSIBLE MULTI-FIG"
                desc_html = '<p style="font-size:10px;color:#856404;margin:4px 0 0 0;">May contain multiple figures — check and rename</p>'
            else:
                badge_bg, badge_col = "#f8d7da", "#721c24"
                badge_txt = "&#9888; UNMATCHED — rename _Fu &#8594; _F&lt;n&gt;"
                desc_html = '<p style="font-size:10px;color:#721c24;margin:4px 0 0 0;">No label found — needs human review</p>'

            cards += f"""
            <div style="border:1px solid #ccc;border-radius:6px;padding:6px;
                        display:flex;flex-direction:column;align-items:center;gap:4px;">
                <div style="background:{badge_bg};color:{badge_col};border:1px solid {badge_col};
                            border-radius:4px;padding:3px 8px;font-size:11px;font-weight:bold;
                            width:100%;text-align:center;">
                    {badge_txt}
                </div>
                <div style="font-size:9px;color:#666;text-align:center;word-break:break-all;">
                    {row["output"]}
                </div>
                {img_tag}
                {desc_html}
            </div>"""

        html = f"""
        <div>
            <h3 style="margin:4px 0;">{patent_id}
                <span style="font-size:12px;font-weight:normal;color:#555;">
                    &nbsp;—&nbsp;{len(grp)} crops &nbsp;|&nbsp;
                    <span style="color:#155724;">{n_ok} matched</span> &nbsp;|&nbsp;
                    <span style="color:#721c24;">{n_fu} unmatched</span>
                </span>
            </h3>
            <details style="margin-bottom:8px;">
                <summary style="cursor:pointer;color:#555;font-size:11px;">Description ({len(fig_descs)} FIG. entries)</summary>
                <p style="font-size:11px;color:#333;max-width:900px;">
                    {str(desc_lookup.get(patent_id,""))[:1200]}
                </p>
            </details>
            <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;">
                {cards}
            </div>
        </div>"""

        counter.value = (f"<span style='color:grey;font-size:12px;'>"
                         f"Patent {idx+1} / {len(patents)}</span>")
        prev_btn.disabled = (idx == 0)
        next_btn.disabled = (idx == len(patents) - 1)

        with out:
            clear_output(wait=True)
            display(HTML(html))

    def on_prev(_): state["idx"] = max(0, state["idx"]-1); render(state["idx"])
    def on_next(_): state["idx"] = min(len(patents)-1, state["idx"]+1); render(state["idx"])
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)

    nav = widgets.HBox([prev_btn, counter, next_btn],
                       layout=widgets.Layout(align_items="center", gap="12px"))
    display(widgets.VBox([nav, out]))
    render(0)


In [ ]:
# ── Cell 8 — All crops viewer (3-column HTML grid) ───────────────────────────
import re, base64, textwrap
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

matched_dir = Path(cfg["paths"]["matched"])
crops_df    = pd.read_csv(Path(cfg["paths"]["data"]) / "crops_mapping.csv")
desc_df     = pd.read_csv(Path(cfg["paths"]["data"]) / "descriptions.csv")
desc_lookup = desc_df.set_index("patent_id")["description_of_drawings"].to_dict()

def _b64(path):
    return base64.b64encode(path.read_bytes()).decode()

def clean_desc(text, max_chars=1200):
    if not isinstance(text, str): return "(no description)"
    return re.sub(r"\s{2,}", " ", text).strip()[:max_chars]

for patent_id, grp in crops_df.groupby("patent_id"):
    folders = [p for p in matched_dir.iterdir() if p.name.startswith(patent_id)]
    if not folders: continue
    out_dir = folders[0]

    n_ok  = (grp["needs_review"] == False).sum()
    n_fu  = (grp["needs_review"] == True).sum()
    desc  = clean_desc(desc_lookup.get(patent_id, ""))

    cards = ""
    for _, row in grp.iterrows():
        img_path = out_dir / row["output"]
        if not img_path.exists():
            continue
        b64   = _b64(img_path)
        label = row.get("label") or ""
        hint  = row.get("review_hint") or ""
        needs_review = row["needs_review"]

        if not needs_review:
            badge_bg  = "#d4edda"
            badge_col = "#155724"
            badge_txt = f"&#10003; MATCHED &nbsp; {label}"
        elif hint == "possible_multi_fig":
            badge_bg  = "#fff3cd"
            badge_col = "#856404"
            badge_txt = "&#9888; POSSIBLE MULTI-FIG"
        else:
            badge_bg  = "#f8d7da"
            badge_col = "#721c24"
            badge_txt = "&#9888; UNMATCHED — needs review"

        cards += f"""
        <div style="border:1px solid #ccc; border-radius:6px; padding:6px;
                    display:flex; flex-direction:column; align-items:center; gap:4px;">
            <div style="background:{badge_bg}; color:{badge_col};
                        border:1px solid {badge_col}; border-radius:4px;
                        padding:3px 8px; font-size:11px; font-weight:bold;
                        width:100%; text-align:center;">
                {badge_txt}
            </div>
            <div style="font-size:10px; color:#555; text-align:center; word-break:break-all;">
                {row["output"]}
            </div>
            <img src="data:image/png;base64,{b64}"
                 style="max-width:100%; max-height:280px; object-fit:contain;"/>
        </div>"""

    html = f"""
    <div style="margin-bottom:30px;">
        <h3 style="margin:0 0 4px 0;">{patent_id}
            <span style="font-size:13px; font-weight:normal; color:#555;">
                &nbsp;—&nbsp; {len(grp)} crops &nbsp;|&nbsp;
                <span style="color:#155724;">{n_ok} matched</span> &nbsp;|&nbsp;
                <span style="color:#721c24;">{n_fu} unmatched</span>
            </span>
        </h3>
        <details><summary style="cursor:pointer; color:#555; font-size:12px;">Description</summary>
            <p style="font-size:11px; color:#333; max-width:900px;">{desc}</p>
        </details>
        <div style="display:grid; grid-template-columns:repeat(3,1fr); gap:10px; margin-top:10px;">
            {cards}
        </div>
    </div>
    <hr/>"""

    display(HTML(html))


## Cell 9 — Duplicate-label audit

Flags two suspicious patterns in `crops_mapping.csv` that signal labelling problems:

1. **Same label on multiple crops of the same sheet** — usually means the sheet has
   side-by-side sub-figures that the vertical splitter kept in one crop, and the OCR
   cascade read the same caption for each row (e.g. three crops all labelled `2A`).
2. **Low label diversity per patent** — many crops but few distinct labels suggests
   the cascade is picking up a neighbouring figure's caption.

These are review hints only — nothing is modified.

In [ ]:
# ── Cell 9 — Duplicate-label audit ────────────────────────────────────────────
import pandas as pd
from pathlib import Path

crops_df = pd.read_csv(Path(cfg["paths"]["data"]) / "crops_mapping.csv")
labelled = crops_df[crops_df["needs_review"] == False].copy()

if labelled.empty:
    print("No labelled crops yet — run the pipeline (Cell 4) first.")
else:
    # 1) Same label repeated on the same sheet
    dup_sheet = (
        labelled.groupby(["patent_id", "original", "label"])
        .size().rename("count").reset_index()
        .query("count > 1")
        .sort_values("count", ascending=False)
    )
    print(f"Same label on multiple crops of one sheet: {len(dup_sheet)} cases")
    if not dup_sheet.empty:
        display(dup_sheet.head(20))

    # 2) Label diversity per patent (crops vs distinct labels)
    diversity = (
        labelled.groupby("patent_id")
        .agg(crops=("output", "count"), distinct_labels=("label", "nunique"))
        .assign(ratio=lambda d: (d["distinct_labels"] / d["crops"]).round(2))
        .sort_values("ratio")
    )
    low = diversity[diversity["ratio"] < 0.7]
    print(f"\nPatents with low label diversity (<70% unique): {len(low)}")
    if not low.empty:
        display(low)

    # 3) Boxes flagged as possibly containing multiple merged sub-figures
    # (e.g. FIG.8A/8B/8C wrapped in one YOLO box — see review_hint in doclayout_matcher.py).
    # Flag only — crop/label is untouched; just helps reviewers spot these fast.
    if "review_hint" in crops_df.columns:
        multi_fig = crops_df[crops_df["review_hint"] == "possible_multi_fig"]
        print(f"\nFlagged as possible multi-figure merges: {len(multi_fig)}")
        if not multi_fig.empty:
            display(multi_fig[["patent_id", "original", "output", "label", "needs_review"]])

In [ ]:
# ── Collect batch outputs → batch_00/ ─────────────────────────────────────────
# Copies this batch's matched crops + data CSVs into a self-contained folder
# for validation. When you're happy and run all patents, skip this cell.

import shutil
from pathlib import Path

BATCH_OUT = Path(cfg["paths"]["base"]) / "batch_00"

# 1. matched crops for this batch
matched_src = Path(cfg["paths"]["matched"])
matched_dst = BATCH_OUT / "matched"
matched_dst.mkdir(parents=True, exist_ok=True)
for pid in run_df["patent_id"]:
    src = matched_src / pid
    if src.exists():
        dst = matched_dst / pid
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)

# 2. data CSVs
data_src = Path(cfg["paths"]["data"])
data_dst = BATCH_OUT / "data"
data_dst.mkdir(parents=True, exist_ok=True)
for csv_name in ["descriptions.csv", "crops_mapping.csv", "needs_human_review.csv"]:
    p = data_src / csv_name
    if p.exists():
        shutil.copy2(p, data_dst / csv_name)

print(f"Batch outputs collected → {BATCH_OUT}")
print(f"  matched/ : {len(list(matched_dst.iterdir()))} patent folders")
print(f"  data/    : {[f.name for f in data_dst.iterdir()]}")